In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from IPython.display import display
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor

from analysis.src.data import complete_case_mask
from analysis.src.io import experiment_paths, load_data
from analysis.src.plotting import configure_plotting
from analysis.src.constants import COMPONENTS, COVARIATES, OUTCOMES

In [ ]:
configure_plotting()

In [ ]:
OUT = experiment_paths('robustness/regression_diagnostics')
DISEASES = OUTCOMES

data = load_data()

In [ ]:
# Check collinearity
vif_sample = data.dropna(subset = [*COMPONENTS, 'walkability_score', *COVARIATES])
predictors = sm.add_constant(vif_sample[['walkability_score', *COVARIATES]])

vif_data = pd.DataFrame(
    {'Feature': predictors.columns, 'VIF': [variance_inflation_factor(predictors.values, i) for i in range(predictors.shape[1])]}
).query("Feature != 'const'")

vif_data.to_csv(OUT.vif_csv, index = False)

In [ ]:
# Fit primary models and examine residual patterns
diagnostic_models = {}
heteroskedasticity_rows = []

fig, axes = plt.subplots(2, 4, figsize = (16, 8), constrained_layout = True)

for ax, outcome in zip(axes.flat, DISEASES):
    model_data = data.loc[complete_case_mask(data, outcome, COVARIATES)].copy()
    formula = f'{outcome} ~ walkability_score + ' + ' + '.join(COVARIATES)
    model = smf.ols(formula, data = model_data).fit()
    diagnostic_models[outcome] = {'model': model, 'data': model_data, 'formula': formula}

    sns.scatterplot(x = model.fittedvalues, y = model.resid, alpha = 0.25, s = 10, ax = ax)
    ax.axhline(0, color = 'red', linestyle = '--', linewidth = 1)
    ax.set(title = outcome.removeprefix('qof_'), xlabel = 'Fitted values', ylabel = 'Residuals')

    bp_lm, bp_lm_p, bp_f, bp_f_p = het_breuschpagan(model.resid, model.model.exog)
    heteroskedasticity_rows.append(
        {
            'outcome': outcome,
            'n_lsoas': int(model.nobs),
            'bp_lm_statistic': bp_lm,
            'bp_lm_p_value': bp_lm_p,
            'bp_f_statistic': bp_f,
            'bp_f_p_value': bp_f_p
        }
    )

fig.savefig(OUT.residuals_png, dpi = 300, bbox_inches = 'tight')

plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize = (16, 8), constrained_layout = True)

for ax, outcome in zip(axes.flat, DISEASES):
    model = diagnostic_models[outcome]['model']
    sns.histplot(model.resid, kde = True, ax = ax)
    ax.set(title = outcome.removeprefix('qof_'), xlabel = 'Residuals', ylabel = 'Count')

fig.savefig(OUT.residual_distributions_png, dpi = 300, bbox_inches = 'tight')

plt.show()

In [ ]:
heteroskedasticity_results = pd.DataFrame(heteroskedasticity_rows)
heteroskedasticity_results.to_csv(OUT.heteroskedasticity_csv, index = False)

display(heteroskedasticity_results.round(4))

In [ ]:
# Compare conventional and HC3 uncertainty
hc3_rows = []
for outcome, specification in diagnostic_models.items():
    model = specification['model']
    hc3_model = model.get_robustcov_results(cov_type = 'HC3')
    idx = model.model.exog_names.index('walkability_score')
    robust_ci = hc3_model.conf_int()[idx]
    ols_ci = model.conf_int().loc['walkability_score']
    
    hc3_rows.append(
        {
            'outcome': outcome,
            'ols_coefficient': model.params['walkability_score'],
            'ols_p_value': model.pvalues['walkability_score'],
            'ols_ci_lower': ols_ci.iloc[0],
            'ols_ci_upper': ols_ci.iloc[1],
            'hc3_coefficient': hc3_model.params[idx],
            'hc3_p_value': hc3_model.pvalues[idx],
            'hc3_ci_lower': robust_ci[0],
            'hc3_ci_upper': robust_ci[1]
        }
    )

robust_results = pd.DataFrame(hc3_rows)
robust_results.to_csv(OUT.hc3_models_csv, index = False)

display(robust_results.round(4))

In [ ]:
# Refit after excluding influential observations
cooks_rows = []

for outcome, specification in diagnostic_models.items():
    model = specification['model']
    model_data = specification['data']

    influence = model.get_influence()
    outliers = influence.cooks_distance[0] > (4 / len(model_data))

    refitted_model = smf.ols(specification['formula'], data = model_data.loc[~outliers]).fit()
    clean_ci = refitted_model.conf_int().loc['walkability_score']

    cooks_rows.append(
        {
            'outcome': outcome,
            'n_lsoas': len(model_data),
            'n_influential_lsoas': int(outliers.sum()),
            'influential_percent': 100 * outliers.mean(),
            'full_coefficient': model.params['walkability_score'],
            'excluding_influential_coefficient': refitted_model.params['walkability_score'],
            'excluding_influential_p_value': refitted_model.pvalues['walkability_score'],
            'excluding_influential_ci_lower': clean_ci.iloc[0],
            'excluding_influential_ci_upper': clean_ci.iloc[1]
        }
    )

cooks_results = pd.DataFrame(cooks_rows)
cooks_results.to_csv(OUT.influential_observations_csv, index = False)

display(cooks_results.round(4))